# NLP project: small-model successors on U-MIRAGE (366-item sample)

All code is in this notebook: prompts and parser (same as the API runs), the Hugging Face runner and the scorers. They are copied from
`scripts/run_eval.py`, `run_local_hf.py`, `score.py` and `score_extra.py`. Only **data** comes from `DATA_DIR`, which contains
`share/nlp_project_sample366.jsonl` and `data/`:
**(a)** upload `nlp_project_data.zip` when asked;
**(b)** unzip it on Google Drive and set `DATA_DIR`. **Recommended**: new results are written to `<DATA_DIR>/results_rerun/` (the included `results/` are never touched) and survive disconnects;
**(c)** set a local path, or run from the replication package root.

| paper row | successor run here | HF model id | precision |
|---|---|---|---|
| Qwen 2.5 0.5B (q, Ollama Q4_K_M) | Qwen3.5 0.8B, quantized | `Qwen/Qwen3.5-0.8B` | bitsandbytes 4-bit NF4 (thinking disabled) |
| LLaMA 3.2 1B q (Ollama default tag = Q8_0) | Llama 3.2 1B Instruct, quantized | `meta-llama/Llama-3.2-1B-Instruct` | bitsandbytes 4-bit NF4 |
| LLaMA 3.2 1B (HF) | Llama 3.2 1B Instruct, full precision | `meta-llama/Llama-3.2-1B-Instruct` | bf16 (fp16 on GPUs without bf16, e.g. T4) |

Settings: original Urdu questions, en/ru/ur × zs/umcot, temperature 0.1 (top_p 1, top_k off), max 2,048 new tokens, shuffled with seed 42.

**Before you start:** *Runtime > Change runtime type > GPU* (L4/A100 recommended). Accept the Llama 3.2 licence at
https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct, create a read token, and add it as Colab secret `HF_TOKEN`. Never paste it into a cell.
Then *Run all*. Re-running resumes (skips done items). Expected time is about 1.5 h on an L4 (our run: 84 min on an NVIDIA L4, bf16) and 1.5–2.5 h on T4.

Our own outputs from this notebook are included in the package under `results/<tag>/`, and our executed copy (with all outputs) is
`notebooks/executed/nlp_project_small_models_executed.ipynb`. To check our numbers without a GPU, use the score-only mode of
`nlp_project_api_models.ipynb` or `python scripts/reproduce_tables.py`, which cover all six models.

**Step 1: Choose what to run.** Sets which models to run, whether to do a 5-item smoke test first, the batch size and the output-token limit. The defaults are the settings we used.

*Input:* nothing. *Output:* `MODELS`, `RUN_SMOKE`, `RUN_FULL`, `BATCH_SIZE`, `MAX_TOKENS`, `DATA_DIR`.

In [ ]:
# ---- settings ----
DATA_DIR = ''             # '' = auto-detect / upload; or e.g. '/content/drive/MyDrive/nlp_project_data'
RUN_SMOKE = True          # first 5 items x 6 runs per model, into smoke_results/
SMOKE_ITEMS = 5
RUN_FULL = True           # full runs: 3 models x 2,196 calls
BATCH_SIZE = 32           # lower to 8-16 on CUDA out-of-memory
MAX_TOKENS = 2048
INSTALL_DEPS = True       # pip-install the pinned GPU stack
REQUIRE_GPU = True        # set False only for a slow CPU test
MODELS = [  # (hf model id, results tag, quant); FULL becomes bf16 or fp16
    ('Qwen/Qwen3.5-0.8B',                'qwen3.5-0.8b-bnb4',          'bnb4'),
    ('meta-llama/Llama-3.2-1B-Instruct', 'llama3.2-1b-instruct-bnb4',  'bnb4'),
    ('meta-llama/Llama-3.2-1B-Instruct', 'llama3.2-1b-instruct-FULL',  'none'),
]

## Data folder

**Step 2: Find the data.** Looks for the data folder (it must contain `share/nlp_project_sample366.jsonl`): the package root, a Drive folder, or a zip you upload in Colab. All later steps read from this folder, so the notebook itself holds only code.

*Input:* `DATA_DIR` or the uploaded `nlp_project_data.zip`. *Output:* `DATA_DIR` / `ROOT` pointing at the data folder.

In [ ]:
# Find the data folder: it must contain share/nlp_project_sample366.jsonl (plus data/, and results/ for score-only).
import os, sys, zipfile
IN_COLAB = 'google.colab' in sys.modules

def is_data_dir(d):
    return bool(d) and os.path.isfile(os.path.join(d, 'share', 'nlp_project_sample366.jsonl'))

if IN_COLAB and DATA_DIR.startswith('/content/drive'):          # (b) folder on Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
here = os.getcwd()
candidates = [DATA_DIR, here, os.path.join(here, 'nlp_project_data'),          # (c) local path / package root
              os.path.dirname(here), os.path.join(os.path.dirname(here), 'nlp_project_data'),
              '/content/nlp_project_data']                                     # (a) uploaded earlier this session
found = next((os.path.abspath(d) for d in candidates if is_data_dir(d)), None)
if found is None and IN_COLAB:                                    # (a) upload the data zip now
    from google.colab import files
    print('Upload nlp_project_data.zip (or nlp_project_replication_package.zip)')
    uploaded = files.upload()
    zname = next(n for n in uploaded if n.endswith('.zip'))
    zipfile.ZipFile(zname).extractall('/content')
    found = next((d for d in ['/content/nlp_project_data', '/content/nlp_project_replication_package'] if is_data_dir(d)), None)
assert is_data_dir(found), 'Data folder not found: set DATA_DIR to the folder that contains share/, data/ (and results/).'
DATA_DIR = found
ROOT = DATA_DIR          # the code below (copied from scripts/) resolves all data paths relative to ROOT
print('data folder:', DATA_DIR)

**Step 3: Install the GPU stack and check the GPU.** Installs the pinned Hugging Face / bitsandbytes versions and checks that a GPU is present. It also decides the full-precision dtype: bf16 when the GPU supports it (L4, A100), otherwise fp16 (T4).

*Input:* a GPU runtime. *Output:* `FULL_DTYPE`; the printout names the GPU and library versions.

In [ ]:
# ---- dependencies + hardware ----
if INSTALL_DEPS:
    %pip install -q transformers==5.24.0 accelerate==1.16.0 bitsandbytes==0.50.2 huggingface_hub==2.4.0 tokenizers==0.23.3 scipy==1.18.1 numpy==2.5.5
    %pip install -q flash-linear-attention
    # optional: fast kernels for Qwen3.5 linear attention. If the install fails, a correct but slower PyTorch fallback is used.
import torch, transformers
if REQUIRE_GPU:
    assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > GPU'
FULL_DTYPE = 'bf16' if (not torch.cuda.is_available() or torch.cuda.is_bf16_supported()) else 'fp16'
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU', '| full-precision dtype:', FULL_DTYPE,
      '| transformers', transformers.__version__)

## Code
#### 1. Prompts (U-MIRAGE Figure 1 wording) and the answer parser

**Step 4: Define the prompts and the answer parser.** Defines U-MIRAGE's zero-shot and "CoT" prompts (Figure 1 wording, verbatim) and the parser that pulls the answer letter out of a reply. Every model uses this same code, so the comparisons are fair.

*Input:* nothing. *Output:* the functions `build_prompt` and `parse_answer`.

In [ ]:
# --- from scripts/run_eval.py (verbatim; the CLI scripts use the same code) ---
import argparse, glob, json, os, random, re, sys, time, datetime as dt


import requests


LANG = {'en': 'English', 'ru': 'Roman Urdu', 'ur': 'Urdu'}


# U-MIRAGE Figure 1, verbatim wording.
ZS = ("This is a medical multiple-choice question written in {language}\n"
      "Question: {question}\nOptions: {options}\n"
      "Select the single correct option (A, B, C, D). Only return the letter corresponding to the correct answer. No explanation.")


# U-MIRAGE's own "CoT" prompt (kept for reference; it forbids written reasoning). Use --conds umcot to run it.
UMCOT = ("This is a medical multiple-choice question written in {language}\n"
         "Question: {question}\nOptions: {options}\n"
         "Let's think step-by-step to reason about the correct answer based on medical knowledge. After reasoning, "
         "select the single best option (A, B, C, or D). Return only the final answer letter (A, B, C, or D) without any explanation.")


# Our written-CoT condition: same frame, reasoning is written out, answer on the last line.
COT = ("This is a medical multiple-choice question written in {language}\n"
       "Question: {question}\nOptions: {options}\n"
       "Let's think step-by-step to reason about the correct answer based on medical knowledge. Write out your reasoning, "
       "then select the single best option (A, B, C, or D). End your reply with a final line of the form 'Answer: X', "
       "where X is the option letter.")


TEMPLATES = {'zs': ZS, 'cot': COT, 'umcot': UMCOT}


def fmt_options(opts):
    return '\n'.join(f'{k}: {v}' for k, v in sorted(opts.items()))


def build_prompt(item, script, cond):
    q = item[f'question_{script}']
    opts = item.get(f'options_{script}')
    if opts is None:
        raise KeyError(f'no options_{script}')
    return TEMPLATES[cond].format(language=LANG[script], question=q, options=fmt_options(opts))


# ---------------------------------------------------------------- answer parsing
PARSER_VERSION = 2


UR_LETTER = {'ا': 'A', 'اے': 'A', 'بی': 'B', 'ب': 'B', 'سی': 'C', 'ڈی': 'D', 'د': 'D'}


def parse_answer(text, valid, cond):
    """Return (letter or None, method)."""
    if not text:
        return None, 'empty'
    t = text.strip()
    valid = set(valid)
    pats = [r'(?:final\s+)?answer\s*(?:is)?\s*[:：\-]?\s*\**\(?([A-D])\)?\b',
            r'(?:jawab|javab|جواب)\s*[:：\-]?\s*\**\(?([A-D])\)?\b']
    if cond in ('cot', 'umcot'):
        lines = [l for l in t.splitlines() if l.strip()]
        last = lines[-1] if lines else ''
        for p in pats:
            m = re.search(p, last, re.I)
            if m and m.group(1).upper() in valid:
                return m.group(1).upper(), 'last_line'
        for p in pats:
            ms = list(re.finditer(p, t, re.I))
            if ms and ms[-1].group(1).upper() in valid:
                return ms[-1].group(1).upper(), 'answer_anywhere'
    core = re.sub(r'[\s\*\.\)\(:"`\'\[\]]', '', t)
    if len(core) > 1:  # model ignored 'only the letter': look at the final line, then explicit conclusions
        lines = [l for l in t.splitlines() if l.strip()]
        last_letters = set(re.findall(r'(?<![A-Za-z])([A-D])(?![A-Za-z])', lines[-1])) if lines else set()
        if len(last_letters) == 1 and next(iter(last_letters)) in valid:
            return last_letters.pop(), 'last_line_letter'
        concl = r"(?:correct\s+(?:option|answer|choice)\s+is|answer\s+is|let'?s\s+(?:output|select|choose|go\s+with)|i\s+(?:choose|select)|best\s+(?:option|answer)\s+is)\s*[:\-]?\s*\**\s*(?:option\s*)?\(?([A-D])\)?(?![A-Za-z])"
        ms = [m for m in re.finditer(concl, t, re.I) if m.group(1).upper() in valid]
        if ms:
            return ms[-1].group(1).upper(), 'conclusion_phrase'
    if core.upper() in valid and len(core) == 1:
        return core.upper(), 'bare_letter'
    m = re.match(r'^\W*(?:option\s*)?([A-D])\b', t, re.I)
    if m and m.group(1).upper() in valid and (len(t) < 40 or cond == 'zs'):
        return m.group(1).upper(), 'leading_letter'
    for p in pats:
        ms = list(re.finditer(p, t, re.I))
        if ms and ms[-1].group(1).upper() in valid:
            return ms[-1].group(1).upper(), 'answer_anywhere'
    if core in UR_LETTER and UR_LETTER[core] in valid:
        return UR_LETTER[core], 'urdu_letter'
    if cond == 'zs':
        letters = set(re.findall(r'\b([A-D])\b', t))
        if len(letters) == 1 and next(iter(letters)) in valid:
            return letters.pop(), 'single_letter_in_text'
    return None, 'unparsed'

#### 2. Hugging Face runner
Batched generation and the same record format as the API runner. Resumable: skips uids already in each file and appends.

**Step 5: Define the Hugging Face runner.** Loads a model from Hugging Face (4-bit NF4 or full precision), generates answers in batches with the same prompts and parser, and writes records in the same format as the API runner. Items already done are skipped.

*Input:* model id, precision, the sample file. *Output:* the function `run_local_hf_main`; when run it writes `<outdir>/<tag>/<script>_<cond>.jsonl`.

In [ ]:
# --- from scripts/run_local_hf.py (verbatim; the CLI scripts use the same code) ---
import argparse, json, os, sys, time, datetime as dt


import numpy as np


def load_items(args):
    items = [json.loads(l) for l in open(args.items)]
    if args.ids:
        import csv
        keep = {r['uid'] for r in csv.DictReader(open(args.ids))}
        items = [it for it in items if it['uid'] in keep]
    if args.limit:
        items = items[:args.limit]
    if args.order == 'shuffle':  # same as run_eval.py
        perm = np.random.RandomState(args.seed).permutation(len(items))
        items = [items[i] for i in perm]
    # Urdu options: the shared sample already carries options_ur; otherwise merge them like run_eval.py
    if any('options_ur' not in it for it in items) and os.path.exists(args.ur_options):
        uo = {j['uid']: j['options_ur'] for j in map(json.loads, open(args.ur_options))}
        for it in items:
            if it['uid'] in uo and 'options_ur' not in it:
                it['options_ur'] = uo[it['uid']]
    if not args.ur_original_questions:
        qf = os.path.join(ROOT, 'data/questions_ur_fixed.jsonl')
        if os.path.exists(qf):
            fx = {j['uid']: j['question_ur_fixed'] for j in map(json.loads, open(qf))}
            for it in items:
                if it['uid'] in fx:
                    it['question_ur'] = fx[it['uid']]; it['ur_question_fixed'] = True
    return items


def load_model(args):
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    dtype = {'bf16': torch.bfloat16, 'fp16': torch.float16, 'fp32': torch.float32}[args.dtype]
    kw = dict(dtype=dtype, token=os.environ.get('HF_TOKEN') or None)
    if args.quant == 'bnb4':
        from transformers import BitsAndBytesConfig
        kw['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                                       bnb_4bit_compute_dtype=dtype, bnb_4bit_use_double_quant=False)
        kw['device_map'] = 'auto'
    elif torch.cuda.is_available():
        kw['device_map'] = 'cuda'
    tok = AutoTokenizer.from_pretrained(args.model_id, token=kw['token'])
    tok.padding_side = 'left'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    try:
        model = AutoModelForCausalLM.from_pretrained(args.model_id, **kw)
    except (ValueError, KeyError) as e:  # multimodal checkpoints -> image-text-to-text class
        print('AutoModelForCausalLM failed, trying AutoModelForImageTextToText:', str(e)[:200])
        from transformers import AutoModelForImageTextToText
        model = AutoModelForImageTextToText.from_pretrained(args.model_id, **kw)
    model.eval()
    return tok, model


def chat_text(tok, prompt):
    msgs = [{'role': 'user', 'content': prompt}]
    try:  # Qwen3/3.5: hard-disable thinking; other templates ignore the extra kwarg
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


def run_local_hf_main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument('--model-id', required=True, help='Hugging Face repo id')
    ap.add_argument('--tag', required=True, help='results/<tag>/ folder name, e.g. llama3.2-1b-instruct-bf16')
    ap.add_argument('--quant', choices=['none', 'bnb4'], default='none', help='bnb4 = bitsandbytes 4-bit NF4 (the paper\'s "q" rows)')
    ap.add_argument('--dtype', choices=['bf16', 'fp16', 'fp32'], default='bf16')
    ap.add_argument('--items', default=os.path.join(ROOT, 'share/nlp_project_sample366.jsonl'))
    ap.add_argument('--ur-options', default=os.path.join(ROOT, 'data/options_ur.jsonl'))
    ap.add_argument('--ur-original-questions', action='store_true')
    ap.add_argument('--ids', default=None)
    ap.add_argument('--scripts', nargs='+', default=['en', 'ru', 'ur'])
    ap.add_argument('--conds', nargs='+', default=['zs', 'umcot'])
    ap.add_argument('--temperature', type=float, default=0.1)
    ap.add_argument('--max-tokens', type=int, default=2048)
    ap.add_argument('--batch-size', type=int, default=16)
    ap.add_argument('--order', choices=['file', 'shuffle'], default='shuffle')
    ap.add_argument('--seed', type=int, default=42)
    ap.add_argument('--limit', type=int, default=None)
    ap.add_argument('--max-calls', type=int, default=None)
    ap.add_argument('--outdir', default=os.path.join(ROOT, 'results'))
    args = ap.parse_args(argv)

    import torch
    items = load_items(args)
    mdir = os.path.join(args.outdir, args.tag)
    os.makedirs(mdir, exist_ok=True)
    jobs = {}
    for s in args.scripts:
        for c in args.conds:
            path = os.path.join(mdir, f'{s}_{c}.jsonl')
            done = {json.loads(l)['uid'] for l in open(path) if l.strip()} if os.path.exists(path) else set()
            jobs[(s, c)] = (path, [it for it in items if it['uid'] not in done and it.get(f'options_{s}')])
            print(f'{s}/{c}: {len(done)} done, {len(jobs[(s, c)][1])} to go', flush=True)
    queues = {k: list(v[1]) for k, v in jobs.items()}
    order = []
    while any(queues.values()):  # interleave (same as run_eval --interleave)
        for k in list(queues):
            if queues[k]:
                order.append((k, queues[k].pop(0)))
    if args.max_calls:
        order = order[:args.max_calls]
    if not order:
        print('nothing to do'); return

    import transformers
    tok, model = load_model(args)
    dev = next(model.parameters()).device
    rev = getattr(model.config, '_commit_hash', None)
    if not rev:
        try:
            from huggingface_hub import HfApi
            rev = HfApi().model_info(args.model_id, token=os.environ.get('HF_TOKEN') or None).sha
        except Exception as e:
            rev = f'unknown ({type(e).__name__})'
    gen_kw = dict(max_new_tokens=args.max_tokens, pad_token_id=tok.pad_token_id)
    if args.temperature > 0:  # override checkpoint generation_config defaults (top_k/top_p/penalties) for parity
        gen_kw.update(do_sample=True, temperature=args.temperature, top_p=1.0, top_k=0, repetition_penalty=1.0)
    else:
        gen_kw.update(do_sample=False)
    eos = {int(x) for x in np.atleast_1d(getattr(model.generation_config, 'eos_token_id', None) or [])} | {tok.eos_token_id}
    hw = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
    print(f'model {args.model_id} rev={rev} quant={args.quant} dtype={args.dtype} device={dev} ({hw}) | {len(order)} calls', flush=True)
    n_ok, t_all = 0, time.time()
    for b0 in range(0, len(order), args.batch_size):
        batch = order[b0:b0 + args.batch_size]
        prompts = [build_prompt(it, s, c) for (s, c), it in batch]
        texts = [chat_text(tok, p) for p in prompts]
        enc = tok(texts, return_tensors='pt', padding=True, add_special_tokens=False).to(dev)
        torch.manual_seed(args.seed + b0)
        t0 = dt.datetime.now(dt.timezone.utc); tt = time.time()
        with torch.no_grad():
            out = model.generate(**enc, **gen_kw)
        lat = time.time() - tt
        t1 = dt.datetime.now(dt.timezone.utc)
        gen = out[:, enc['input_ids'].shape[1]:]
        for i, (((s, c), it), prompt) in enumerate(zip(batch, prompts)):
            ids = gen[i].tolist()
            n_out, hit_eos = 0, False
            for t in ids:
                if t in eos:
                    n_out += 1; hit_eos = True; break
                if t == tok.pad_token_id:
                    break
                n_out += 1
            text = tok.decode(ids[:n_out], skip_special_tokens=True).strip()
            n_in = int(enc['attention_mask'][i].sum())
            pred, how = parse_answer(text, it['options_en'].keys(), c)
            rec = dict(uid=it['uid'], source=it['source'], script=s, cond=c, gold=it['answer'], pred=pred,
                       correct=pred == it['answer'], parse=how,
                       ur_question_fixed=it.get('ur_question_fixed', False) if s == 'ur' else None,
                       response=text, finish_reason='stop' if hit_eos else 'length',
                       model=args.tag, model_version=f'{args.model_id}@{rev}', response_id=None,
                       usage={'promptTokenCount': n_in, 'candidatesTokenCount': n_out, 'thoughtsTokenCount': 0,
                              'totalTokenCount': n_in + n_out},
                       temperature=args.temperature,
                       thinking_level='disabled (enable_thinking=False)' if 'qwen' in args.model_id.lower() else None,
                       thinking_budget=None, retries=0, prompt=prompt, provider='hf-transformers',
                       reasoning_effort=None, temperature_sent=args.temperature, max_tokens=args.max_tokens,
                       errors=[], system_fingerprint=f'transformers {transformers.__version__}, torch {torch.__version__}, {hw}',
                       service_tier=None, refusal=None, quantization=args.quant, dtype=args.dtype, hf_model_id=args.model_id,
                       sampling=dict(do_sample=gen_kw.get('do_sample'), top_p=gen_kw.get('top_p'), top_k=gen_kw.get('top_k'), seed=args.seed + b0),
                       ur_question_truncated_in_original=it.get('ur_question_truncated_in_original', it.get('ur_truncated')) if s == 'ur' else None,
                       latency_s=round(lat / len(batch), 3), batch_latency_s=round(lat, 3), batch_size=len(batch),
                       t_start=t0.isoformat(), t_end=t1.isoformat(), parser_version=PARSER_VERSION)
            with open(jobs[(s, c)][0], 'a') as f:
                f.write(json.dumps(rec, ensure_ascii=False) + '\n')
            n_ok += 1
        el = time.time() - t_all
        print(f'{n_ok}/{len(order)} done | batch {len(batch)} in {lat:.1f}s | elapsed {el/60:.1f} min | '
              f'ETA {(len(order)-n_ok)*el/n_ok/60:.1f} min | last: {s}/{c} gold={it["answer"]} pred={pred} ({how})', flush=True)
    print('done;', n_ok, 'calls')

#### 3. Scorers

**Step 6: Define the main scorer.** Re-parses every raw reply and computes accuracy, paired EN-RU / EN-UR / RU-UR gaps with 95% bootstrap intervals and exact McNemar tests, accuracy per source dataset, and which script the model replied in.

*Input:* raw `.jsonl` results. *Output:* the function `score_main`; when run it writes `<model>/scores_sample366.md`.

In [ ]:
# --- from scripts/score.py (verbatim; the CLI scripts use the same code) ---
import argparse, glob, json, os, re


import numpy as np, sys


from scipy.stats import binomtest


RU_WORDS = set('hai hain ka ki ke mein se aur nahi nahin kya yeh ye woh wo ko ho hota hoti hote jata jati liye bhi tha thi sakta sakti kar karta karti par lekin kyunke kyun agar jo ek sab zyada kam wala wali'.split())


EN_WORDS = set('the is are of and to in that with for this which it be as by on not or an was'.split())


def reply_script(text):
    t = re.sub(r'(?im)^\s*answer\s*:.*$', '', text or '')
    ar = len(re.findall(r'[\u0600-\u06FF]', t)); la = len(re.findall(r'[A-Za-z]', t))
    if ar + la == 0:
        return 'none'
    if ar / (ar + la) > 0.5:
        return 'ur'
    w = re.findall(r'[a-z]+', t.lower())
    ru, en = sum(x in RU_WORDS for x in w), sum(x in EN_WORDS for x in w)
    return 'ru' if ru > en else 'en'


def boot_ci(diff, B=10000, seed=0):
    rng = np.random.default_rng(seed); n = len(diff)
    m = diff[rng.integers(0, n, size=(B, n))].mean(1)
    return np.percentile(m, [2.5, 97.5])


def mcnemar(a, b):
    b01, b10 = int(((a == 1) & (b == 0)).sum()), int(((a == 0) & (b == 1)).sum())
    p = binomtest(b01, b01 + b10, 0.5).pvalue if b01 + b10 else 1.0
    return b01, b10, p


def score_main(argv=None):
    ap = argparse.ArgumentParser(); ap.add_argument('--model', required=True)
    ap.add_argument('--ids', default=None, help='csv with a uid column: score only these items')
    ap.add_argument('--outdir', default=os.path.join(ROOT, 'results'))
    args = ap.parse_args(argv)
    mdir = os.path.join(args.outdir, args.model.replace('/', '_'))
    SUBSRC = {j['uid']: j['source'] for j in map(json.loads, open(os.path.join(ROOT, 'data/subset_items.jsonl')))}
    VALID = {j['uid']: sorted(j['options_en']) for j in map(json.loads, open(os.path.join(ROOT, 'data/subset_items.jsonl')))}
    KEEP = None
    if args.ids:
        import csv
        KEEP = {r['uid'] for r in csv.DictReader(open(args.ids))}
        SUBSRC = {u: v for u, v in SUBSRC.items() if u in KEEP}
    R = {}
    for f in sorted(glob.glob(os.path.join(mdir, '*_*.jsonl'))):
        s, c = os.path.basename(f)[:-6].split('_', 1)
        recs = {}
        for l in open(f):
            r = json.loads(l)
            valid = VALID.get(r['uid']) or 'ABCD'  # re-parse raw reply with current parser
            r['pred'], r['parse'] = parse_answer(r['response'], valid, r['cond']); r['correct'] = r['pred'] == r['gold']
            if KEEP is None or r['uid'] in KEEP:
                recs[r['uid']] = r  # last write wins
        R[(s, c)] = recs
    out = [f'# Scores for {args.model} (answers re-parsed from raw replies, parser v{PARSER_VERSION})\n']
    out.append('| script | cond | n | accuracy | source-weighted acc* | parse fail | zs non-compliant (not a bare letter) | mean in tok | mean out tok (visible+thinking) |\n|---|---|---|---|---|---|---|---|---|')
    for (s, c), recs in sorted(R.items()):
        v = list(recs.values()); n = len(v)
        acc = np.mean([r['correct'] for r in v]); pf = sum(r['pred'] is None for r in v)
        ui = np.mean([(r['usage'] or {}).get('promptTokenCount', (r['usage'] or {}).get('prompt_tokens', 0)) for r in v])
        uo = np.mean([(r['usage'] or {}).get('candidatesTokenCount', 0) + (r['usage'] or {}).get('thoughtsTokenCount', 0)
                      + (r['usage'] or {}).get('completion_tokens', 0) for r in v])  # OpenAI usage keys too
        nc = sum(len(re.sub(r'[\s\*\.\(\)]', '', r['response'] or '')) > 1 for r in v) if c == 'zs' else None
        nc_s = f'{nc} ({100*nc/n:.1f}%)' if nc is not None else '-'
        W = {so: sum(1 for x in SUBSRC.values() if x == so) / len(SUBSRC) for so in set(SUBSRC.values())}
        per = {so: np.mean([r['correct'] for r in v if r['source'] == so]) for so in W if sum(r['source'] == so for r in v) >= 10}
        wacc = sum(W[so] * a for so, a in per.items()) / sum(W[so] for so in per) if per else float('nan')
        out.append(f'| {s} | {c} | {n} | {100*acc:.1f} | {100*wacc:.1f} | {pf} ({100*pf/n:.1f}%) | {nc_s} | {ui:.0f} | {uo:.0f} |')
    out.append('\n*re-weighted to the subset source mix (over sources with >=10 items done), to offset processing-order imbalance.')
    out.append('\n## Paired comparisons (items answered in both; diff = first - second, pp)\n')
    out.append('| cond | pair | n paired | acc 1st | acc 2nd | diff | 95% bootstrap CI (10k) | b01/b10 | McNemar exact p |\n|---|---|---|---|---|---|---|---|---|')
    for c in sorted({c for _, c in R}):
        for a, b in [('en', 'ur'), ('en', 'ru'), ('ru', 'ur')]:
            if (a, c) not in R or (b, c) not in R:
                continue
            ids = sorted(set(R[(a, c)]) & set(R[(b, c)]))
            if len(ids) < 2:
                continue
            x = np.array([R[(a, c)][i]['correct'] for i in ids], int); y = np.array([R[(b, c)][i]['correct'] for i in ids], int)
            lo, hi = boot_ci(x - y); b01, b10, p = mcnemar(x, y)
            out.append(f'| {c} | {a.upper()}-{b.upper()} | {len(ids)} | {100*x.mean():.1f} | {100*y.mean():.1f} | {100*(x-y).mean():+.1f} | [{100*lo:+.1f}, {100*hi:+.1f}] | {b01}/{b10} | {p:.3g} |')
    out.append('\n## Accuracy by source dataset (%; n in brackets)\n')
    srcs = sorted({r['source'] for recs in R.values() for r in recs.values()})
    out.append('| script/cond | ' + ' | '.join(srcs) + ' |\n|---|' + '---|' * len(srcs))
    for (s, c), recs in sorted(R.items()):
        cells = []
        for so in srcs:
            v = [r['correct'] for r in recs.values() if r['source'] == so]
            cells.append(f'{100*np.mean(v):.1f} [{len(v)}]' if v else '-')
        out.append(f'| {s}/{c} | ' + ' | '.join(cells) + ' |')
    cot = [(k, v) for k, v in R.items() if k[1] in ('cot', 'umcot')]
    if cot:
        out.append('\n## Reply script for reasoning outputs (heuristic: Arabic-script share; Roman Urdu vs English by function words)\n')
        out.append('| asked | n | replied en | replied ru | replied ur | % in asked script |\n|---|---|---|---|---|---|')
        for (s, c), recs in sorted(cot):
            sc = [reply_script(r['response']) for r in recs.values()]; n = len(sc)
            out.append(f'| {s}/{c} | {n} | {sc.count("en")} | {sc.count("ru")} | {sc.count("ur")} | {100*sc.count(s)/n:.0f}% |')
    txt = '\n'.join(out) + '\n'
    open(os.path.join(mdir, 'scores' + ('_' + os.path.basename(args.ids).split('_')[0] if args.ids else '') + '.md'), 'w').write(txt); print(txt)

**Step 7: Define the paper-style tables.** Builds the U-MIRAGE Table 4-style accuracy table (with the paper's reference rows), the gap table, the truncated vs complete Urdu question split, and answer-format compliance.

*Input:* raw `.jsonl` results. *Output:* the function `score_extra_main`; writes `<model>/scores_extra_sample366.md` and a combined `summary_*.md`.

In [ ]:
# --- from scripts/score_extra.py (verbatim; the CLI scripts use the same code) ---
import argparse, json, os, re, sys, csv


import numpy as np


CONDS = [('zs', 'Zero-Shot'), ('umcot', 'CoT (U-MIRAGE prompt)')]


SCRIPTS = ['en', 'ru', 'ur']


# U-MIRAGE paper reference (umirage/all_configurations_accuracy.csv; full benchmark, "CoT" = their with_reasoning prompt)
PAPER_ALL = {  # U-MIRAGE all_configurations_accuracy.csv, full 7,663-item benchmark ("CoT" = their with_reasoning prompt)
    'GPT-5 nano [paper]':            {('en', 'zs'): .7962, ('en', 'umcot'): .7865, ('ru', 'zs'): .7502, ('ru', 'umcot'): .7498, ('ur', 'zs'): .7299, ('ur', 'umcot'): .7023},
    'Gemini 2.0 Flash-Lite [paper]': {('en', 'zs'): .7177, ('en', 'umcot'): .7779, ('ru', 'zs'): .6582, ('ru', 'umcot'): .7414, ('ur', 'zs'): .6768, ('ur', 'umcot'): .7013},
    'Gemma 2 2B [paper]':            {('en', 'zs'): .4198, ('en', 'umcot'): .4325, ('ru', 'zs'): .3247, ('ru', 'umcot'): .3500, ('ur', 'zs'): .3124, ('ur', 'umcot'): .3367},
    'Qwen 2.5 0.5B (q) [paper]':     {('en', 'zs'): .2592, ('en', 'umcot'): .2974, ('ru', 'zs'): .2378, ('ru', 'umcot'): .2765, ('ur', 'zs'): .2305, ('ur', 'umcot'): .2721},
    'LLaMA 3.2 1B (q) [paper]':      {('en', 'zs'): .1886, ('en', 'umcot'): .3889, ('ru', 'zs'): .1982, ('ru', 'umcot'): .2400, ('ur', 'zs'): .1883, ('ur', 'umcot'): .1781},
    'LLaMA 3.2 1B [paper]':          {('en', 'zs'): .4549, ('en', 'umcot'): .4437, ('ru', 'zs'): .3448, ('ru', 'umcot'): .3616, ('ur', 'zs'): .3364, ('ur', 'umcot'): .3124},
}


PAPER_SETS = {'small': ['Qwen 2.5 0.5B (q) [paper]', 'LLaMA 3.2 1B (q) [paper]', 'LLaMA 3.2 1B [paper]'],
              'api': ['GPT-5 nano [paper]', 'Gemini 2.0 Flash-Lite [paper]', 'Gemma 2 2B [paper]'],
              'all': list(PAPER_ALL)}


# which U-MIRAGE paper row each of our runs succeeds (for the side-by-side table)
SUCCESSOR_OF = {'gpt-6-luna': 'GPT-5 nano [paper]', 'gemini-3.5-flash-lite': 'Gemini 2.0 Flash-Lite [paper]',
                'gemma-4-26b-a4b-it': 'Gemma 2 2B [paper]', 'qwen3.5-0.8b-bnb4': 'Qwen 2.5 0.5B (q) [paper]',
                'llama3.2-1b-instruct-bnb4': 'LLaMA 3.2 1B (q) [paper]', 'llama3.2-1b-instruct-bf16': 'LLaMA 3.2 1B [paper]'}


def load(mdir, keep):
    R = {}
    for s in SCRIPTS:
        for c, _ in CONDS:
            f = os.path.join(mdir, f'{s}_{c}.jsonl')
            if not os.path.exists(f):
                continue
            recs = {}
            for l in open(f):
                if not l.strip():
                    continue
                r = json.loads(l)
                if keep and r['uid'] not in keep:
                    continue
                r['pred'], r['parse'] = parse_answer(r['response'], 'ABCD' if r['gold'] in 'ABCD' else r['gold'], r['cond'])
                r['correct'] = r['pred'] == r['gold']
                recs[r['uid']] = r
            R[(s, c)] = recs
    return R


def paired(R, a, b, c, ids=None):
    if (a, c) not in R or (b, c) not in R:
        return None
    common = set(R[(a, c)]) & set(R[(b, c)])
    if ids is not None:
        common &= ids
    common = sorted(common)
    if len(common) < 2:
        return None
    x = np.array([R[(a, c)][i]['correct'] for i in common], int); y = np.array([R[(b, c)][i]['correct'] for i in common], int)
    lo, hi = boot_ci(x - y); b01, b10, p = mcnemar(x, y)
    return len(common), x.mean(), y.mean(), (x - y).mean(), lo, hi, b01, b10, p


def score_extra_main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument('--models', nargs='+', required=True)
    ap.add_argument('--labels', nargs='*', default=None, help='display names, same order as --models')
    ap.add_argument('--outdir', default=os.path.join(ROOT, 'results'))
    ap.add_argument('--items', default=os.path.join(ROOT, 'share/nlp_project_sample366.jsonl'))
    ap.add_argument('--ids', default=os.path.join(ROOT, 'data/sample366_ids.csv'))
    ap.add_argument('--paper-rows', choices=list(PAPER_SETS), default='small')
    ap.add_argument('--summary-name', default='summary_small_models_sample366.md')
    ap.add_argument('--title', default='Small-model successors on the 366-item sample')
    args = ap.parse_args(argv)
    items = {j['uid']: j for j in map(json.loads, open(args.items))}
    keep = {r['uid'] for r in csv.DictReader(open(args.ids))} if args.ids and os.path.exists(args.ids) else set(items)
    trunc = {u for u, j in items.items() if j.get('ur_question_truncated_in_original', j.get('ur_truncated'))}
    labels = args.labels or args.models
    summary = [f'# {args.title} (parser v{PARSER_VERSION})\n',
               '## Paper-style accuracy table (fraction correct, 2 decimals; n per cell in the per-model files)\n',
               '| Model | EN Zero-Shot | EN CoT | RU Zero-Shot | RU CoT | UR Zero-Shot | UR CoT |', '|---|---|---|---|---|---|---|']
    gaps_rows, trunc_rows, comp_rows, pct_rows, pct_of = [], [], [], [], {}
    for tag, lab in zip(args.models, labels):
        mdir = os.path.join(args.outdir, tag)
        R = load(mdir, keep)
        out = [f'# Extra scores for {lab} ({tag}), parser v{PARSER_VERSION}\n',
               '| script | cond | n | accuracy | 95% bootstrap CI | parse fail | zs not bare letter | hit max_tokens |', '|---|---|---|---|---|---|---|---|']
        cells = []
        for s in SCRIPTS:
            for c, _ in CONDS:
                v = list(R.get((s, c), {}).values())
                if not v:
                    cells.append('-'); continue
                x = np.array([r['correct'] for r in v], float)
                rng = np.random.default_rng(0); bs = x[rng.integers(0, len(x), size=(10000, len(x)))].mean(1)
                lo, hi = np.percentile(bs, [2.5, 97.5])
                pf = sum(r['pred'] is None for r in v)
                nb = sum(len(re.sub(r'[\s\*\.\(\)]', '', r['response'] or '')) > 1 for r in v) if c == 'zs' else None
                ln = sum(r.get('finish_reason') in ('length', 'MAX_TOKENS') for r in v)
                out.append(f'| {s} | {c} | {len(v)} | {x.mean():.2f} | [{lo:.2f}, {hi:.2f}] | {pf} ({100*pf/len(v):.1f}%) | '
                           f'{"-" if nb is None else f"{nb} ({100*nb/len(v):.1f}%)"} | {ln} ({100*ln/len(v):.1f}%) |')
                cells.append(f'{x.mean():.2f}' + ('' if len(v) == len(keep) else f' (n={len(v)})'))
                comp_rows.append(f'| {lab} | {s}/{c} | {len(v)} | {pf} ({100*pf/len(v):.1f}%) | {"-" if nb is None else f"{100*nb/len(v):.1f}%"} | {100*ln/len(v):.1f}% |')
        summary.append(f'| {lab} | ' + ' | '.join(cells) + ' |')
        pct_of[tag] = [f'{100*np.mean([r["correct"] for r in R[(s, c)].values()]):.2f}' if R.get((s, c)) else '-' for s in SCRIPTS for c, _ in CONDS]
        pct_rows.append(f'| {lab} | ' + ' | '.join(pct_of[tag]) + ' |')
        out += ['\n## Paired gaps (accuracy in %, diff = first - second in percentage points; bootstrap 10k; exact McNemar)\n',
                '| cond | pair | n | acc 1st | acc 2nd | diff | 95% CI | b01/b10 | p |', '|---|---|---|---|---|---|---|---|---|']
        for c, cn in CONDS:
            for a, b in [('en', 'ru'), ('en', 'ur'), ('ru', 'ur')]:
                g = paired(R, a, b, c)
                if g:
                    n, xa, xb, d, lo, hi, b01, b10, p = g
                    row = f'| {c} | {a.upper()}-{b.upper()} | {n} | {100*xa:.2f} | {100*xb:.2f} | {100*d:+.2f} | [{100*lo:+.2f}, {100*hi:+.2f}] | {b01}/{b10} | {p:.3g} |'
                    out.append(row); gaps_rows.append(f'| {lab} ' + row)
        out += ['\n## Urdu: questions truncated in the released U-MIRAGE CSV vs complete (original questions used)\n',
                '| cond | subset | n | UR acc | EN acc (same items) | EN-UR | 95% CI | McNemar p |', '|---|---|---|---|---|---|---|---|']
        for c, cn in CONDS:
            for nm, ids in [('truncated', trunc & keep), ('complete', keep - trunc)]:
                g = paired(R, 'en', 'ur', c, ids)
                if g:
                    n, xa, xb, d, lo, hi, b01, b10, p = g
                    row = f'| {c} | {nm} | {n} | {100*xb:.2f} | {100*xa:.2f} | {100*d:+.2f} | [{100*lo:+.2f}, {100*hi:+.2f}] | {p:.3g} |'
                    out.append(row); trunc_rows.append(f'| {lab} ' + row)
        txt = '\n'.join(out) + '\n'
        open(os.path.join(mdir, 'scores_extra_sample366.md'), 'w').write(txt)
        print(txt)
    summary.append('| *U-MIRAGE paper reference rows (full benchmark, 7,663 items):* | | | | | | |')
    for lab in PAPER_SETS[args.paper_rows]:
        d = PAPER_ALL[lab]
        summary.append(f'| {lab} | ' + ' | '.join(f'{d[(s, c)]:.2f}' for s in SCRIPTS for c, _ in CONDS) + ' |')
    summary += ['\n## Accuracy in % (2 decimals)\n', '| Model | EN ZS | EN CoT | RU ZS | RU CoT | UR ZS | UR CoT |', '|---|---|---|---|---|---|---|'] + pct_rows
    pairs = [(SUCCESSOR_OF[t], t, lab) for t, lab in zip(args.models, labels) if SUCCESSOR_OF.get(t) in PAPER_SETS[args.paper_rows]]
    order = list(PAPER_ALL)
    pairs.sort(key=lambda x: order.index(x[0]))
    if pairs:
        summary += ['\n## Paper rows vs our successors (accuracy in %, 2 decimals; each paper row directly above its successor)\n',
                    'Paper = U-MIRAGE, full benchmark (7,663 items). Ours = a random 366-item sample (seed 42), so the rows are not item-matched. '
                    'Gemma 4 26B-A4B is not size-matched to Gemma 2 2B (26B total / ~4B active parameters vs 2B).\n',
                    '| Model | EN ZS | EN CoT | RU ZS | RU CoT | UR ZS | UR CoT |', '|---|---|---|---|---|---|---|']
        for prow, t, lab in pairs:
            summary.append(f'| {prow.replace(" [paper]", "")} [paper] | ' + ' | '.join(f'{100*PAPER_ALL[prow][(s, c)]:.2f}' for s in SCRIPTS for c, _ in CONDS) + ' |')
            summary.append(f'| {lab} [ours] | ' + ' | '.join(pct_of[t]) + ' |')
    summary += ['\n## Paired gaps (accuracy %, diff in pp)\n', '| model | cond | pair | n | acc 1st | acc 2nd | diff | 95% CI | b01/b10 | McNemar p |',
                '|---|---|---|---|---|---|---|---|---|---|'] + gaps_rows
    summary += ['\n## Truncated vs complete Urdu questions (accuracy %, diff in pp)\n', '| model | cond | subset | n | UR acc | EN acc | EN-UR | 95% CI | McNemar p |',
                '|---|---|---|---|---|---|---|---|---|'] + trunc_rows
    summary += ['\n## Compliance\n', '| model | run | n | parse fail | zs not bare letter | hit max_tokens |', '|---|---|---|---|---|---|'] + comp_rows
    txt = '\n'.join(summary) + '\n'
    open(os.path.join(args.outdir, args.summary_name), 'w').write(txt)
    print(txt)

## Hugging Face token (gated Llama)

**Step 8: Hugging Face token for Llama.** Llama 3.2 is a gated model. This reads `HF_TOKEN` from Colab secrets (or asks for it with hidden input) and checks access. The token is never printed.

*Input:* Colab secret `HF_TOKEN`, licence accepted on Hugging Face. *Output:* "Llama access OK", or a message saying what to fix.

In [ ]:
needs_llama = any('llama' in mid.lower() for mid, _, _ in MODELS)
if needs_llama and not os.environ.get('HF_TOKEN'):
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN') or ''
    except Exception:
        from getpass import getpass
        os.environ['HF_TOKEN'] = getpass('HF token (read access; input hidden): ')
if needs_llama:
    from huggingface_hub import HfApi
    try:
        HfApi().model_info('meta-llama/Llama-3.2-1B-Instruct', token=os.environ['HF_TOKEN']); print('Llama access OK')
    except Exception as e:
        print('!! No access to meta-llama/Llama-3.2-1B-Instruct: accept the licence on its HF page and check the token.', type(e).__name__)

## Runs

**Step 9: Smoke test.** Defines the three runs and, if `RUN_SMOKE`, runs 5 items per language and prompt for each model. This checks model loading, the chat template, parsing and speed before the long run.

*Input:* models from Hugging Face, the sample file. *Output:* `smoke_results/<tag>/*.jsonl`.

In [ ]:
RUNS = [(mid, tag.replace('FULL', FULL_DTYPE), q, FULL_DTYPE) for mid, tag, q in MODELS]
RESULTS_DIR = os.path.join(DATA_DIR, 'results_rerun')   # new runs; the included results/ stay untouched
COMMON = ['--items', os.path.join(DATA_DIR, 'share/nlp_project_sample366.jsonl'), '--ur-original-questions',
          '--scripts', 'en', 'ru', 'ur', '--conds', 'zs', 'umcot', '--temperature', '0.1',
          '--max-tokens', str(MAX_TOKENS), '--seed', '42', '--batch-size', str(BATCH_SIZE)]
def hf_run(mid, tag, q, dtype, outdir, extra=()):
    print(f'\n=== {tag} ({mid}, quant={q}, dtype={dtype}) -> {outdir}/{tag}')
    run_local_hf_main(['--model-id', mid, '--tag', tag, '--quant', q, '--dtype', dtype, *COMMON, '--outdir', outdir, *extra])
    import gc; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if RUN_SMOKE:   # checks loading, chat template, parsing and speed
    for r in RUNS:
        hf_run(*r, outdir=os.path.join(DATA_DIR, 'smoke_results'), extra=['--limit', str(SMOKE_ITEMS)])

**Step 10: Full runs.** Runs all 366 items × 3 languages × 2 prompts (2,196 calls) for each model. If the runtime disconnects, re-run this cell and it continues.

*Input:* models, the sample file. *Output:* `<RESULTS_DIR>/<tag>/*.jsonl`: `results_rerun/` in the current notebook; our original run (executed copy in `notebooks/executed/`) wrote to `results/`.

In [ ]:
# ---- full runs (resumable: re-run this cell after a disconnect) ----
if RUN_FULL:
    for r in RUNS:
        hf_run(*r, outdir=RESULTS_DIR)

## Scores

**Step 11: Score the small models.** Scores each model and builds the combined table with the U-MIRAGE paper's small-model rows for comparison.

*Input:* the raw outputs from the runs. *Output:* `<tag>/scores_sample366.md`, `<tag>/scores_extra_sample366.md`, `summary_small_models_sample366.md`.

In [ ]:
# score_main per model -> <dir>/<tag>/scores_sample366.md; score_extra_main -> combined summary (paper rows: small models)
import contextlib, io
from IPython.display import Markdown, display
SCORE_DIR = RESULTS_DIR if RUN_FULL else os.path.join(DATA_DIR, 'smoke_results')
LAB = {'qwen3.5-0.8b-bnb4': 'Qwen3.5 0.8B (q, NF4)', 'llama3.2-1b-instruct-bnb4': 'Llama 3.2 1B Instruct (q, NF4)'}
tags = [tag for _, tag, _, _ in RUNS if os.path.isdir(os.path.join(SCORE_DIR, tag))]
with contextlib.redirect_stdout(io.StringIO()):
    for tag in tags:
        score_main(['--model', tag, '--outdir', SCORE_DIR, '--ids', os.path.join(DATA_DIR, 'data/sample366_ids.csv')])
    score_extra_main(['--outdir', SCORE_DIR, '--models', *tags, '--labels', *[LAB.get(t, f'Llama 3.2 1B Instruct ({FULL_DTYPE})' if 'llama' in t else t) for t in tags],
                      '--paper-rows', 'small', '--summary-name', 'summary_small_models_sample366.md',
                      '--title', 'Small-model successors on the 366-item sample'])
display(Markdown(open(os.path.join(SCORE_DIR, 'summary_small_models_sample366.md')).read()))

**Step 12: Package the results.** Zips the raw outputs, score files and smoke results so they can be downloaded and sent back.

*Input:* the results folders. *Output:* `nlp_project_small_models_results.zip`.

In [ ]:
# ---- zip results (raw jsonl + scores) for sending back ----
import shutil, tempfile
pack = tempfile.mkdtemp()
for tag in tags:
    shutil.copytree(os.path.join(SCORE_DIR, tag), os.path.join(pack, 'results', tag))
shutil.copy(os.path.join(SCORE_DIR, 'summary_small_models_sample366.md'), os.path.join(pack, 'results'))
if os.path.isdir(os.path.join(DATA_DIR, 'smoke_results')):
    shutil.copytree(os.path.join(DATA_DIR, 'smoke_results'), os.path.join(pack, 'smoke_results'))
z = shutil.make_archive(os.path.join(DATA_DIR, 'nlp_project_small_models_results'), 'zip', pack)
print('wrote', z)
if IN_COLAB:
    from google.colab import files; files.download(z)